# Updated Fast Marching and scribble demo

This is a **new 2026 educational demonstration**, prepared by Codex for Aaron Wetzler. Only the local Eikonal stencil is derived from the recovered, authorship-unverified CutOut-related prototype. The heapq solver, boundaries, multiple seeds, synthetic image, image-to-cost rule and label competition are new. This is **not the Android app or an original research result**.

Run from a clone containing `updated/fmm.py`. No package installation, data download or camera is required. In Colab, clone the public repository in a separate setup cell and change into its `updated/` directory before running these cells. The code cells below do not perform network access.

In [ ]:
from pathlib import Path
import sys
roots = [Path.cwd(), Path.cwd().parent, Path.cwd() / "updated", Path.cwd() / "cutout-android" / "updated"]
updated = next((p for p in roots if (p / "fmm.py").is_file()), None)
if updated is None:
    raise RuntimeError("Clone cutout-android and run from its updated/ or notebooks/ directory")
sys.path.insert(0, str(updated.resolve()))
from fmm import fast_marching
from scribble_demo import procedural_scene, segment, intersection_over_union, render_svg
print("Local updated modules loaded; no original app or external data was loaded.")


## Adjust the experiment

Change `SIZE` or `EDGE_WEIGHT`, then run the remaining cells again. The new cost rule is `1 + EDGE_WEIGHT * local intensity change`. Larger values make it more expensive for an arrival front to cross an image edge. The foreground scribble is pink; the background scribbles are cyan. Both seed sets and the textured disk are procedural.

In [ ]:
SIZE = 48  # integer 16..256; smaller values run faster
EDGE_WEIGHT = 24.0  # nonnegative; try 0, 8, 24, 60
scene = procedural_scene(SIZE)
result = segment(scene, EDGE_WEIGHT)
print(f"Synthetic {SIZE}x{SIZE} disk; edge weight {EDGE_WEIGHT:g}")
print(f"Synthetic mask IoU: {intersection_over_union(result.mask, scene.truth):.6f}")
print("This score describes this procedural disk only, not original app accuracy.")


## See the image, scribbles, arrivals and new mask

Arrival colors share one normalization across both fields. A pixel becomes foreground when its foreground arrival is finite and no later than the background arrival. This two-label composition was newly added; `testScribbles.m` does not provide a recovered final segmentation implementation.

In [ ]:
svg = render_svg(scene, result)
try:
    from IPython.display import SVG, display
except ImportError:
    print(f"Four-panel SVG generated ({len(svg):,} characters); Jupyter displays it inline.")
else:
    display(SVG(svg))


## Compare with the edge barrier removed

This changes a new demonstration assumption, rather than restoring or evaluating the historical Android workflow. The mask can improve or degrade as scribbles and costs change. A procedural example cannot establish accuracy on photographs.

In [ ]:
no_edges = segment(scene, 0.0)
print(f"No edge cost: {intersection_over_union(no_edges.mask, scene.truth):.6f} synthetic IoU")
print(f"Chosen edge cost: {intersection_over_union(result.mask, scene.truth):.6f} synthetic IoU")


## A numerical check independent of the picture

On a unit-cost grid, axis steps cost one. The recovered first-order discrete stencil gives `1 + 1/sqrt(2)` at the first diagonal, rather than exact Euclidean `sqrt(2)`. The complete tests additionally cover symmetry, cost scaling, obstacles, boundary seeds and invalid inputs. An optional C++ harness compares 1,005 local cases with the hash-verified original update function; it does not run the original heap, MEX app or Android application.

In [ ]:
import math
times = fast_marching([[1.0] * 3 for _ in range(3)], [(0, 0)])
assert times[0] == [0.0, 1.0, 2.0]
assert math.isclose(times[1][1], 1 + 1 / math.sqrt(2), rel_tol=1e-14)
print("Analytic axis/diagonal check passed.")
